어느정돈 성공을 했다. 77%의 성공률과 그중 26개에대해서는 완벽한 평행사변형을 찾아냈다. 그래서 최종 적용을 위한 마지막 3번째 4점찾기 코드로 들어간다. 성공한 것들을 벤치마킹해서 보정하는것이다.

#1. 4점 찾기

In [ ]:
from pathlib import Path
import random
import math
import time
import itertools
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 실행 옵션 (사용자 원본 경로 및 변수 100% 복구)
# =========================================================
# 파일 이름으로 찾을 최상위 경로 설정 (원본 유지)
SEARCH_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0. ROI BOX 원본/이름변경 전_to 0412")

# [학생] 새로 이름이 변경된 이미지들이 들어있는 폴더
SAVEFILE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0. ROI BOX 원본/이름변경 후_to 0412")

# [선생님] 50장의 'ㅇ' 표시와 좌표가 있는 이전 결과 엑셀
V2_RESULT_CSV = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 전_to 0412/_circle_anchor_test_v2/per_image_results_v2._1.csv")

# 결과 저장 루트
OUT_ROOT = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412")

RANDOM_SEED = 42
MAX_WORKERS = 8
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# 1. ROI / 전처리 / 후보 검출 (사용자 원본 로직 유지)
# =========================================================
LEFT_ROI_RATIO = 0.28
UPSCALE = 1.5
BLUR_KSIZE = 5
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)
TOP_CUT_RATIO = 0.05
BOTTOM_CUT_RATIO = 0.05

THRESH_METHOD = "otsu_inv"
MIN_AREA = 120
MAX_AREA = 7000
MIN_CIRCULARITY = 0.35
MAX_AXIS_RATIO = 2.4
MIN_DARKNESS_DIFF = 12
RING_MARGIN = 1.7
DEDUP_DIST = 18
MIN_VISIBLE_RATE = 0.25

# =========================================================
# 2. template-driven 보정 규칙 (사용자 원본 로직 유지)
# =========================================================
GOOD_NOTE_VALUE = "ㅇ"
PARTIAL_MATCH_DIST_RATIO = 0.16
MAX_TEMPLATE_SHIFT_RATIO = 0.08
TEMPLATE_SHIFT_STEPS = 5
REQUIRE_MATCH_FOR_CORRECTED = 2

FONT = cv2.FONT_HERSHEY_SIMPLEX

# =========================================================
# 유틸 (사용자 원본 유지)
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def list_image_files_recursive(root_dir):
    root_dir = Path(root_dir)
    return sorted([p for p in root_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])


def dist(p1, p2):
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))


def slope(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.degrees(math.atan2(dy, dx))


def order_points_tl_tr_br_bl(points):
    pts = np.array(points, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def circularity_from_contour(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri <= 0:
        return 0.0
    return float(4.0 * math.pi * area / (peri * peri))


def preprocess_gray(gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    out = clahe.apply(gray)
    if UPSCALE != 1.0:
        h, w = out.shape[:2]
        out = cv2.resize(out, (int(w * UPSCALE), int(h * UPSCALE)), interpolation=cv2.INTER_CUBIC)
    out = cv2.GaussianBlur(out, (BLUR_KSIZE, BLUR_KSIZE), 0)
    return out


def make_binary_for_dark_holes(proc_gray):
    if THRESH_METHOD == "otsu_inv":
        _, bw = cv2.threshold(proc_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        bw = cv2.adaptiveThreshold(
            proc_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 31, 7
        )
    kernel = np.ones((3, 3), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=1)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    return bw


def compute_darkness_score(gray, cx, cy, radius):
    h, w = gray.shape[:2]
    yy, xx = np.indices((h, w))
    rr2 = (xx - cx) ** 2 + (yy - cy) ** 2
    inner = rr2 <= (radius ** 2)
    outer = (rr2 <= ((radius * RING_MARGIN) ** 2)) & (~inner)
    if inner.sum() < 20 or outer.sum() < 20:
        return None, None, 0.0
    inner_mean = float(gray[inner].mean())
    outer_mean = float(gray[outer].mean())
    diff = outer_mean - inner_mean
    return inner_mean, outer_mean, diff


def candidate_visible_rate(x, y, w, h, roi_w, roi_h):
    box_area = max(1, w * h)
    clipped_x1 = max(0, x)
    clipped_y1 = max(0, y)
    clipped_x2 = min(roi_w, x + w)
    clipped_y2 = min(roi_h, y + h)
    clipped_area = max(0, clipped_x2 - clipped_x1) * max(0, clipped_y2 - clipped_y1)
    return clipped_area / box_area


def deduplicate_candidates(candidates, dist_thresh=18):
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda x: x["score_individual"], reverse=True)
    kept = []
    for c in candidates:
        ok = True
        for k in kept:
            if dist((c["cx"], c["cy"]), (k["cx"], k["cy"])) < dist_thresh:
                ok = False
                break
        if ok:
            kept.append(c)
    return kept


def detect_dark_hole_candidates(roi_bgr):
    gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    proc = preprocess_gray(gray)
    bw = make_binary_for_dark_holes(proc)
    contours, _ = cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    ph, pw = proc.shape[:2]

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA or area > MAX_AREA:
            continue
        peri = cv2.arcLength(cnt, True)
        if peri <= 0 or len(cnt) < 5:
            continue

        circ = circularity_from_contour(cnt)
        if circ < MIN_CIRCULARITY:
            continue

        (cx, cy), (ma, mi), _ = cv2.fitEllipse(cnt)
        major = max(ma, mi)
        minor = min(ma, mi)
        if minor <= 0:
            continue
        axis_ratio = major / minor
        if axis_ratio > MAX_AXIS_RATIO:
            continue

        radius = 0.25 * (ma + mi)
        x, y, w, h = cv2.boundingRect(cnt)
        visible_rate = candidate_visible_rate(x, y, w, h, pw, ph)
        if visible_rate < MIN_VISIBLE_RATE:
            continue

        inner_mean, outer_mean, dark_diff = compute_darkness_score(proc, cx, cy, radius)
        if dark_diff is None or dark_diff < MIN_DARKNESS_DIFF:
            continue

        roi_h = ph / UPSCALE
        cy_real = cy / UPSCALE
        if cy_real < roi_h * TOP_CUT_RATIO:
            continue
        if cy_real > roi_h * (1 - BOTTOM_CUT_RATIO):
            continue

        candidates.append({
            "cx": float(cx / UPSCALE),
            "cy": float(cy / UPSCALE),
            "radius": float(radius / UPSCALE),
            "circularity": float(circ),
            "axis_ratio": float(axis_ratio),
            "inner_mean": float(inner_mean),
            "outer_mean": float(outer_mean),
            "dark_diff": float(dark_diff),
            "visible_rate": float(visible_rate),
            "score_individual": float(dark_diff + 20 * circ - 5 * (axis_ratio - 1.0) + 8 * visible_rate),
        })

    return deduplicate_candidates(candidates, DEDUP_DIST)


# =========================================================
# template 생성 / 적용 (After 기반 유지, 엑셀 경로 참조)
# =========================================================
def load_good_template_points(csv_path: Path, roi_ratio: float):
    if not csv_path.exists():
        raise FileNotFoundError(f"v2 결과 CSV가 없습니다: {csv_path}")

    df = pd.read_csv(csv_path)
    if "notes" not in df.columns:
        raise ValueError("v2 CSV에 notes 컬럼이 없습니다. good 케이스 표시가 필요합니다.")

    good = df[df["notes"].astype(str) == GOOD_NOTE_VALUE].copy()
    if len(good) == 0:
        raise ValueError("notes=='ㅇ' 인 good 케이스가 없습니다.")

    req_cols = [f"p{i}_{xy}" for i in range(1, 5) for xy in ["x", "y"]]
    miss = [c for c in req_cols if c not in good.columns]
    if miss:
        raise ValueError(f"다음 컬럼이 없습니다: {miss}")

    templates = []
    for _, row in good.iterrows():
        # [핵심 수정] 예시 파일은 엑셀 내 image_path로 직접 가서 확인
        img_path = Path(row["image_path"])
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        roi_w = int(w * roi_ratio)
        pts = np.array([
            [row["p1_x"], row["p1_y"]],
            [row["p2_x"], row["p2_y"]],
            [row["p3_x"], row["p3_y"]],
            [row["p4_x"], row["p4_y"]],
        ], dtype=np.float32)
        pts = order_points_tl_tr_br_bl(pts)
        norm = pts.copy()
        norm[:, 0] = norm[:, 0] / max(roi_w, 1)
        norm[:, 1] = norm[:, 1] / max(h, 1)
        templates.append(norm)

    if len(templates) == 0:
        raise ValueError("good 케이스에서 template를 만들 수 없습니다.")

    arr = np.stack(templates, axis=0)
    mean_template = arr.mean(axis=0)
    std_template = arr.std(axis=0)
    return mean_template, std_template, len(templates)


def template_to_image_points(template_norm, roi_w, img_h, dx=0.0, dy=0.0):
    pts = np.array(template_norm, dtype=np.float32).copy()
    pts[:, 0] *= roi_w
    pts[:, 1] *= img_h
    pts[:, 0] += dx
    pts[:, 1] += dy
    return pts


def match_candidates_to_template(candidates, template_pts, roi_w, img_h):
    if len(candidates) == 0:
        return [], 0.0
    diag = math.hypot(roi_w, img_h)
    thr = PARTIAL_MATCH_DIST_RATIO * diag

    used = set()
    matched = []
    total_dist = 0.0
    for i, tp in enumerate(template_pts):
        best_j = None
        best_d = None
        for j, c in enumerate(candidates):
            if j in used:
                continue
            d = dist((float(tp[0]), float(tp[1])), (c["cx"], c["cy"]))
            if best_d is None or d < best_d:
                best_d = d
                best_j = j
        if best_j is not None and best_d is not None and best_d <= thr:
            used.add(best_j)
            matched.append((i, candidates[best_j], best_d))
            total_dist += best_d
    return matched, total_dist


def find_best_shifted_template(candidates, template_norm, roi_w, img_h):
    dx_step = roi_w * MAX_TEMPLATE_SHIFT_RATIO / max(TEMPLATE_SHIFT_STEPS - 1, 1)
    dy_step = img_h * MAX_TEMPLATE_SHIFT_RATIO / max(TEMPLATE_SHIFT_STEPS - 1, 1)

    offsets = []
    half = TEMPLATE_SHIFT_STEPS // 2
    for ix in range(-half, half + 1):
        for iy in range(-half, half + 1):
            offsets.append((ix * dx_step, iy * dy_step))

    best = None
    for dx, dy in offsets:
        template_pts = template_to_image_points(template_norm, roi_w, img_h, dx=dx, dy=dy)
        matched, total_dist = match_candidates_to_template(candidates, template_pts, roi_w, img_h)
        n = len(matched)
        avg_dist = total_dist / max(n, 1)
        score = n * 1000 - avg_dist
        if (best is None) or (score > best["score"]):
            best = {
                "score": score,
                "dx": dx,
                "dy": dy,
                "template_pts": template_pts,
                "matched": matched,
                "matched_count": n,
                "avg_dist": avg_dist,
            }
    return best


def corrected_points_from_template(template_pts, matched_list):
    final_pts = template_pts.copy().astype(np.float32)
    for idx, cand, _ in matched_list:
        final_pts[idx, 0] = cand["cx"]
        final_pts[idx, 1] = cand["cy"]
    final_pts = order_points_tl_tr_br_bl(final_pts)
    return final_pts

# =========================================================
# 시각화 (Before 버전 복구)
# =========================================================
def draw_overlay(image_bgr, roi_bgr, all_candidates, final_pts, source, matched_count, save_path):
    vis = image_bgr.copy()
    h, _ = vis.shape[:2]
    roi_w = roi_bgr.shape[1]
    cv2.rectangle(vis, (0, 0), (roi_w - 1, h - 1), (0, 255, 255), 2)

    for c in all_candidates:
        cx = int(round(c["cx"]))
        cy = int(round(c["cy"]))
        rr = int(round(max(4, c["radius"])))
        cv2.circle(vis, (cx, cy), rr, (180, 180, 180), 1)
        cv2.circle(vis, (cx, cy), 2, (0, 255, 255), -1)

    pts = np.array(final_pts, dtype=np.float32)
    pts_i = pts.astype(int)
    for i, p in enumerate(pts_i, start=1):
        cv2.circle(vis, tuple(p), 10, (0, 255, 0), 2)
        cv2.circle(vis, tuple(p), 3, (0, 0, 255), -1)
        cv2.putText(vis, str(i), (p[0] + 6, p[1] - 6), FONT, 0.65, (255, 0, 0), 2, cv2.LINE_AA)
    for i in range(4):
        cv2.line(vis, tuple(pts_i[i]), tuple(pts_i[(i + 1) % 4]), (255, 255, 0), 2)

    color = (0, 255, 0) if source == "corrected" else (0, 0, 255)
    txt = f"source={source} matched={matched_count}"
    cv2.putText(vis, txt, (10, h - 18), FONT, 0.55, color, 2, cv2.LINE_AA)
    cv2.imwrite(str(save_path), vis)


# =========================================================
# 개별 이미지 처리 (After 로직 유지)
# =========================================================
def process_one_image(row, template_norm):
    image_path = Path(row["image_path"])
    out = {
        "subfolder": row["subfolder"],
        "image_path": str(image_path),
        "image_name": row["image_name"],
        "n_candidates": 0,
        "source": "template_only",
        "matched_count": 0,
        "avg_match_dist": np.nan,
        "template_dx": np.nan,
        "template_dy": np.nan,
        "notes": np.nan,
        "p1_x": np.nan, "p1_y": np.nan,
        "p2_x": np.nan, "p2_y": np.nan,
        "p3_x": np.nan, "p3_y": np.nan,
        "p4_x": np.nan, "p4_y": np.nan,
    }

    img = cv2.imread(str(image_path))
    if img is None:
        out["notes"] = "image_read_fail"
        return out, None, None, None, None

    h, w = img.shape[:2]
    roi_w = int(w * LEFT_ROI_RATIO)
    roi = img[:, :roi_w].copy()

    candidates = detect_dark_hole_candidates(roi)
    out["n_candidates"] = len(candidates)

    best = find_best_shifted_template(candidates, template_norm, roi_w, h)
    template_pts = best["template_pts"]
    matched = best["matched"]
    matched_count = best["matched_count"]
    avg_dist = best["avg_dist"]
    dx = best["dx"]
    dy = best["dy"]

    if matched_count >= REQUIRE_MATCH_FOR_CORRECTED:
        final_pts = corrected_points_from_template(template_pts, matched)
        source = "corrected"
        notes = f"template_corrected_with_{matched_count}"
    else:
        final_pts = order_points_tl_tr_br_bl(template_pts)
        source = "template_only"
        notes = "template_only"

    out["source"] = source
    out["matched_count"] = matched_count
    out["avg_match_dist"] = avg_dist
    out["template_dx"] = dx
    out["template_dy"] = dy
    out["notes"] = notes

    for i in range(4):
        out[f"p{i+1}_x"] = float(final_pts[i][0])
        out[f"p{i+1}_y"] = float(final_pts[i][1])

    return out, img, roi, candidates, final_pts


# =========================================================
# 메인 실행부 (전수 조사 모드 + 보고서/오버레이 복구)
# =========================================================
def main():
    t0 = time.time()
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    ensure_dir(OUT_ROOT)
    overlay_dir = OUT_ROOT / "overlays"
    ensure_dir(overlay_dir)

    print("[1] 엑셀의 image_path를 참조하여 선생님 이미지들로부터 형태 학습 중...")
    template_norm, template_std, n_good = load_good_template_points(V2_RESULT_CSV, LEFT_ROI_RATIO)
    template_df = pd.DataFrame({
        "point": ["p1", "p2", "p3", "p4"],
        "x_mean_norm": template_norm[:, 0],
        "y_mean_norm": template_norm[:, 1],
        "x_std_norm": template_std[:, 0],
        "y_std_norm": template_std[:, 1],
    })
    template_df.to_csv(OUT_ROOT / "template_summary.csv", index=False, encoding="utf-8-sig")
    print(f"good 케이스 수: {n_good}")

    print(f"[2] {SAVEFILE_DIR} 에서 학생 이미지(전수 조사 대상) 수집 중...")
    image_paths = list_image_files_recursive(SAVEFILE_DIR)
    # 결과 폴더 내부 파일은 제외하는 필터링 유지 (이름만 비교하면 중복될 수 있으니 전체 문자열 비교)
    image_paths = [p for p in image_paths if str(OUT_ROOT) not in str(p)]
    print(f"전체 이미지 수: {len(image_paths):,}")

    if len(image_paths) == 0:
        print("이미지가 없습니다. 경로 확인 필요.")
        return

    all_images_data = []
    for p in image_paths:
        all_images_data.append({
            "subfolder": str(p.parent),
            "image_path": str(p),
            "image_name": p.name,
        })
    all_images_df = pd.DataFrame(all_images_data)

    print("[3] template-driven 방식으로 anchor 4점 생성 중...")
    rows = all_images_df.to_dict("records")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one_image, row, template_norm) for row in rows]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Warp 4점 분석"):
            # Before 버전처럼 모든 반환값을 받아 오버레이 생성
            out, img, roi, candidates, final_pts_for_overlay = fut.result()
            results.append(out)
            if img is not None:
                overlay_name = Path(out["image_name"]).stem + f"__{out['source']}.jpg"
                save_path = overlay_dir / overlay_name
                draw_overlay(img, roi, candidates if candidates is not None else [], final_pts_for_overlay, out["source"], out["matched_count"], save_path)

    # 보고서 생성 로직 (Before 버전 그대로 복구)
    df = pd.DataFrame(results).sort_values(["subfolder", "image_name"]).reset_index(drop=True)
    summary = (
        df["source"].value_counts(dropna=False)
        .rename_axis("source")
        .reset_index(name="count")
    )
    summary["ratio"] = summary["count"] / max(len(df), 1)

    csv_path = OUT_ROOT / "per_image_results_v3.csv"
    xlsx_path = OUT_ROOT / "circle_anchor_test_report_v3.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        template_df.to_excel(writer, sheet_name="template_summary", index=False)
        df.to_excel(writer, sheet_name="per_image_results", index=False)
        summary.to_excel(writer, sheet_name="source_summary", index=False)

    elapsed = time.time() - t0
    print("\n===== 완료 =====")
    print(f"CSV : {csv_path}")
    print(f"XLSX: {xlsx_path}")
    print(f"Overlay 폴더: {overlay_dir}")
    print(f"총 시간: {elapsed/60:.1f}분")
    print(summary)


if __name__ == "__main__":
    main()

[1] 엑셀의 image_path를 참조하여 선생님 이미지들로부터 형태 학습 중...
good 케이스 수: 25
[2] /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0. ROI BOX 원본/이름변경 후_to 0412 에서 학생 이미지(전수 조사 대상) 수집 중...
전체 이미지 수: 2,358
[3] template-driven 방식으로 anchor 4점 생성 중...


Warp 4점 분석:   0%|          | 0/2358 [00:00<?, ?it/s]


===== 완료 =====
CSV : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/per_image_results_v3.csv
XLSX: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/circle_anchor_test_report_v3.xlsx
Overlay 폴더: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/overlays
총 시간: 4.8분
          source  count    ratio
0      corrected   2287  0.96989
1  template_only     71  0.03011


# xlsx 실행 코드

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 경로 설정
input_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/per_image_results_v3.csv"
output_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/per_image_results_vf.csv"

# 2. 데이터 불러오기
df = pd.read_csv(input_path)

# 3. 유틸리티 함수 정의 (벡터 계산용)
def get_dist(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def get_slope(x1, y1, x2, y2):
    return np.degrees(np.arctan2(y2 - y1, x2 - x1))

# 4. 지표 계산 수행
# 길이 및 대각선
df["quad_width_top"] = get_dist(df["p1_x"], df["p1_y"], df["p2_x"], df["p2_y"])
df["quad_width_bottom"] = get_dist(df["p4_x"], df["p4_y"], df["p3_x"], df["p3_y"])
df["quad_height_left"] = get_dist(df["p1_x"], df["p1_y"], df["p4_x"], df["p4_y"])
df["quad_height_right"] = get_dist(df["p2_x"], df["p2_y"], df["p3_x"], df["p3_y"])
df["quad_diag_1"] = get_dist(df["p1_x"], df["p1_y"], df["p3_x"], df["p3_y"])
df["quad_diag_2"] = get_dist(df["p2_x"], df["p2_y"], df["p4_x"], df["p4_y"])

# 각도(기울기)
df["top_slope_deg"] = get_slope(df["p1_x"], df["p1_y"], df["p2_x"], df["p2_y"])
df["bottom_slope_deg"] = get_slope(df["p4_x"], df["p4_y"], df["p3_x"], df["p3_y"])
df["left_slope_deg"] = get_slope(df["p1_x"], df["p1_y"], df["p4_x"], df["p4_y"])
df["right_slope_deg"] = get_slope(df["p2_x"], df["p2_y"], df["p3_x"], df["p3_y"])

# 비율
df["width_ratio"] = df["quad_width_top"] / (df["quad_width_bottom"] + 1e-6)
df["height_ratio"] = df["quad_height_left"] / (df["quad_height_right"] + 1e-6)
df["diag_ratio"] = df["quad_diag_1"] / (df["quad_diag_2"] + 1e-6)

# 면적 (Shoelace formula)
df["quad_area"] = 0.5 * np.abs(
    (df["p1_x"]*df["p2_y"] + df["p2_x"]*df["p3_y"] + df["p3_x"]*df["p4_y"] + df["p4_x"]*df["p1_y"]) -
    (df["p1_y"]*df["p2_x"] + df["p2_y"]*df["p3_x"] + df["p3_y"]*df["p4_x"] + df["p4_y"]*df["p1_x"])
)

# 중심점
df["centroid_x"] = (df["p1_x"] + df["p2_x"] + df["p3_x"] + df["p4_x"]) / 4
df["centroid_y"] = (df["p1_y"] + df["p2_y"] + df["p3_y"] + df["p4_y"]) / 4

# 5. 결과 저장
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"지표 계산 완료. '{output_path}' 파일을 확인하세요.")

Error: Required CSV file not found. Please ensure the full script has run successfully at least once to generate the CSVs. [Errno 2] No such file or directory: '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/1. circle_anchor/이름변경 후_to 0412/template_summary_v3.csv'
